In [1]:
pip install -U transformers accelerate bitsandbytes peft trl datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 57.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


# Import Libraries

In [2]:
import os, json, torch
import pandas as pd
from datasets import Dataset
from collections import Counter
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
)
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, PeftModel, TaskType

import os
os.environ["WANDB_DISABLED"] = "true"


# Mount Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Configuration

In [22]:
Model = "mistralai/Mistral-7B-Instruct-v0.3"
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project"
OUTPUT_DIR = BASE_DIR + "/Drug_Review/mistral_cls_adapter_001"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Train_path = BASE_DIR + "/Drug_Review/train.csv"
Val_path = BASE_DIR + "/Drug_Review/val.csv"
# Test_path = BASE_DIR + "/Drug_Review/test_medical_abstract.csv"

Labels = ["1-2 stars", "3-4 stars", "5-6 stars", "7-8 stars", "9-10 stars"]

In [23]:
train_df = pd.read_csv(Train_path, dtype={"text":str, "label":str})
display(train_df)

,drugName,condition,review,text,labels
0,zoloft,panic disorde,i have taken 25mg for anxiety for 8 years. i w...,zoloft | panic disorde | i have taken 25mg for...,4
1,exenatide,"diabetes, type 2",bydureon helped me get my a1c numbers down. if...,"exenatide | diabetes, type 2 | bydureon helped...",3
2,macrobid,bladder infection,i am on my seventh day on macrobid and still h...,macrobid | bladder infection | i am on my seve...,0
3,percocet,pain,i have severe back pain and leg pain associate...,percocet | pain | i have severe back pain and ...,4
4,ultram,ibromyalgia,this medicine was useful at first then your bo...,ultram | ibromyalgia | this medicine was usefu...,2
...,...,...,...,...,...
22157,sprintec,birth control,i started taking sprintec yesterday. within a ...,sprintec | birth control | i started taking sp...,0
22158,prednisone,gouty arthritis,ive tried most nsaids and non of them work for...,prednisone | gouty arthritis | ive tried most ...,4
22159,docosanol,herpes simplex,"i have had cold sore all my life, and usually ...",docosanol | herpes simplex | i have had cold s...,1
22160,amrix,muscle spasm,i was having muscle spasm and neck pain which ...,amrix | muscle spasm | i was having muscle spa...,4


In [24]:
val_df = pd.read_csv(Val_path, dtype={"text":str, "label":str})
display(val_df)

,drugName,condition,review,text,label
0,amoxicillin clavulanate,skin or soft tissue infection,this med was given as a result of a deep gouge...,amoxicillin clavulanate | skin or soft tissue ...,0
1,gianvi,premenstrual dysphoric disorde,i was given this generic by my pharmacy who di...,gianvi | premenstrual dysphoric disorde | i wa...,0
2,liletta,birth control,i have had the liletta iud in place for about ...,liletta | birth control | i have had the lilet...,0
3,ethinyl estradiol levonorgestrel,birth control,this is my second month on lutera and it has s...,ethinyl estradiol levonorgestrel | birth contr...,2
4,ethinyl estradiol norgestimate,birth control,sprintec is great for me. i have taken it for ...,ethinyl estradiol norgestimate | birth control...,4
...,...,...,...,...,...
5536,mirena,abnormal uterine bleeding,well after i was told i needed mirena for heav...,mirena | abnormal uterine bleeding | well afte...,4
5537,drospirenone ethinyl estradiol,birth control,"ive been on it for 3 months, and its pretty go...",drospirenone ethinyl estradiol | birth control...,3
5538,liraglutide,"diabetes, type 2","so far so good,,,been on it for 11 days have l...","liraglutide | diabetes, type 2 | so far so goo...",4
5539,ethinyl estradiol norgestimate,birth control,the first couple of months were a bit odd: i e...,ethinyl estradiol norgestimate | birth control...,4


In [25]:
# Rename column
train_df.rename(columns={'labels': 'label'}, inplace=True)
val_df.rename(columns={'labels': 'label'}, inplace=True)

# Save back
train_df.to_csv(Train_path, index=False)
val_df.to_csv(Val_path, index=False)

# Data Preparation

In [26]:
def build_example(text, label=None):
  try:
    system = "You are a helpful classifier. Reply with exactly one label from: " + ", ".join(Labels) + "."
    user = f"Text: {text}\nLabel options: " + ", ".join(Labels) + "\nAnswer with one label only."
    return {
        "system": system,
        "user": user,
        "assistant": label or "",
    }
  except:
    print(text, "\n")
    raise

recs = []
def df_to_hf(df, is_train=True): # converts df into a Hugging Face dataset
    recs = []
    try:
      for _, row in df.iterrows():
          recs.append(build_example(row["text"], Labels[row["label"]] if is_train else None))
      return Dataset.from_list(recs)
    except:
      print(row, "\n")
      # return Dataset.from_list(recs)


train_df = pd.read_csv(Train_path)
val_df = pd.read_csv(Val_path)

train_ds = df_to_hf(train_df)
val_ds = df_to_hf(val_df)

In [27]:
train_ds

Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 22162
})

In [28]:
print(train_ds[:2]["assistant"])

['9-10 stars', '7-8 stars']


In [29]:
print(f"[Step 2] Train rows={len(train_df):,}, Val rows={len(val_df):,}")
print(f"[Step 2] Label examples: {train_df['label'].value_counts().to_dict()}")

[Step 2] Train rows=22,162, Val rows=5,541
[Step 2] Label examples: {4: 10618, 3: 4057, 0: 3760, 2: 2093, 1: 1634}


# Tokenizer + Quantization

In [30]:
tokenizer = AutoTokenizer.from_pretrained(Model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

# LoRA Configuration

In [31]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
)

train_cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=50,
    save_steps=1000,
    bf16=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    packing=False,
    dataset_num_proc=4,
    report_to="none",
    logging_dir="./logs",
)


# Training

In [32]:
def formatting_func(example):
    messages = [
        {"role": "system", "content": example["system"]},
        {"role": "user", "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    # Convert chat messages to a single text sequence (no tokenization yet)
    return tokenizer.apply_chat_template(messages, tokenize=False)

trainer = SFTTrainer(
    model=Model,                 # or model name string
    peft_config=peft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=train_cfg,
    formatting_func=formatting_func,  # single-example formatter
)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

...

Truncating train dataset (num_proc=4):   0%|          | 0/22162 [00:00<?, ? examples/s]

Applying formatting function to eval dataset (num_proc=4):   0%|          | 0/5541 [00:00<?, ? examples/s]

Adding EOS to eval dataset (num_proc=4):   0%|          | 0/5541 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=4):   0%|          | 0/5541 [00:00<?, ? examples/s]

Truncating eval dataset (num_proc=4):   0%|          | 0/5541 [00:00<?, ? examples/s]

In [33]:
trainer.train()
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,1.606300,1.586880,1.635765,1502249.000000,0.637094
1000,1.596200,1.570845,1.594135,2996884.000000,0.639064
1500,1.382700,1.577096,1.445113,4487892.000000,0.640637
2000,1.360800,1.568900,1.434777,5979684.000000,0.642566
2500,1.353600,1.560493,1.444946,7476981.000000,0.643645


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


('/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Drug_Review/mistral_cls_adapter_001/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Drug_Review/mistral_cls_adapter_001/special_tokens_map.json',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Drug_Review/mistral_cls_adapter_001/chat_template.jinja',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Drug_Review/mistral_cls_adapter_001/tokenizer.model',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Drug_Review/mistral_cls_adapter_001/added_tokens.json',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Drug_Review/mistral_cls_adapter_001/tokenizer.json')